# FAC_NB_Demo_Libreria_A — código compartido con %run / notebookutils.notebook.run

**Variante 1.** Las funciones viven en el notebook `FAC_NB_Utils` del mismo workspace.

Requisitos: `FAC_NB_Utils` existe en este workspace; el Lakehouse predeterminado de este notebook es
`FAC_LH_Utils` (aquí se escribe la bitácora); `AB_Lakehouse` tiene las tablas de la Sesión 3 (solo lectura).

Qué debe observar el facilitador:
1. `notebook.run` devuelve un texto pero **no** deja las funciones disponibles.
2. `%run` sí las deja disponibles, porque ejecuta el notebook en el contexto actual.
3. Cuánto tarda cada una (celdas de medición; no hay cifras prometidas).

In [ ]:
p_variante = "A_run"
p_lakehouse_origen = "AB_Lakehouse"
p_tabla_transacciones = "ab_silver_transacciones"

## Parte 1 — `notebookutils.notebook.run` (llamada; devuelve el valor de salida)

In [ ]:
import json
import time

t0 = time.perf_counter()
salida = notebookutils.notebook.run("FAC_NB_Utils", 120, {"p_accion": "version"})
t_notebook_run = time.perf_counter() - t0

print("Valor de salida del hijo:", json.loads(salida))
print(f"notebookutils.notebook.run: {t_notebook_run:.2f} s")
verificaciones = {"funciones_visibles_tras_notebook_run": "normalizar_columnas" in globals()}
print("¿normalizar_columnas quedó disponible en este notebook?", verificaciones["funciones_visibles_tras_notebook_run"])  # esperado: False

## Parte 2 — `%run` (mismo contexto; las funciones quedan disponibles)

La medición va en tres celdas porque `%run` es una orden mágica y no se puede envolver en Python:
marca de tiempo → `%run` → diferencia. El resultado incluye el paso entre celdas; repita con la sesión
ya iniciada para no mezclar el arranque de Spark con la medición.

In [ ]:
import time

_t0_run = time.perf_counter()

In [ ]:
%run FAC_NB_Utils

In [ ]:
t_run = time.perf_counter() - _t0_run
tiempos = {"notebook_run_s": round(t_notebook_run, 3), "run_magic_s": round(t_run, 3)}
print(f"%run FAC_NB_Utils: {t_run:.2f} s (incluye el paso entre celdas)")
verificaciones["funciones_visibles_tras_run"] = "normalizar_columnas" in globals()
print("¿normalizar_columnas disponible ahora?", verificaciones["funciones_visibles_tras_run"])   # esperado: True
print("Versión de las funciones:", version())

## Usar las funciones (este bloque es idéntico en las tres variantes)

Las cuatro utilidades de la demo: **normalizar columnas**, **validar un rango de fechas**,
**hash de fila para conciliar** y **escribir en la bitácora**. Los datos son sintéticos.

In [ ]:
import json

datos = [("TX1", "2026-04-01", 100.50, 1), ("TX2", "2026-09-30", 20.00, 2), ("TX3", "2026-10-05", 5.25, 3)]
df_sucio = spark.createDataFrame(datos, ["ID Transacción", "Fecha de Alta (UTC)", "Monto (CRC)", "Monto  CRC"])
df_limpio = normalizar_columnas(df_sucio)
print("Antes:  ", df_sucio.columns)
print("Después:", df_limpio.columns)   # esperado: id_transaccion, fecha_de_alta_utc, monto_crc, monto_crc_2
verificaciones["columnas_normalizadas"] = df_limpio.columns == ["id_transaccion", "fecha_de_alta_utc", "monto_crc", "monto_crc_2"]

In [ ]:
r_ejemplo = validar_rango_fechas(df_limpio, "fecha_de_alta_utc", "2026-04-01", "2026-09-30")
print(r_ejemplo)   # esperado: total 3, fuera_de_rango 1 (2026-10-05), ok False

tx = leer_tabla_delta(spark, p_lakehouse_origen, p_tabla_transacciones)   # solo lectura
r_tx = validar_rango_fechas(tx, "fecha_hora", "2026-04-01", "2026-09-30")
print(r_tx)        # esperado con el dataset del laboratorio: total 72000, nulos 0, fuera_de_rango 0, ok True
verificaciones["rango_ejemplo_fuera_de_rango"] = r_ejemplo["fuera_de_rango"]
verificaciones["rango_tx"] = {k: r_tx[k] for k in ("total", "nulos", "fuera_de_rango", "ok")}

In [ ]:
columnas_hash = ["transaccion_id", "cuenta_id", "direccion", "canal", "monto"]
tx_h = hash_fila(tx, columnas_hash)

# SHA-256 esperado de las tres primeras transacciones (dataset sintético del laboratorio).
ESPERADO = {
    "TX00000001": "4d01750fa7d9891609de2ddfaca0af9f435d9037944b6d3d49948a8cbea76a50",
    "TX00000002": "967eb9eeb586c00246cccfccdd521ce6f72750fa24275b6850caf7ad9697ff99",
    "TX00000003": "bcec1f1f890d7e1d57e575cdf6fa6a5902f8c7d2061ff70e0e5fef85e0aff819",
}
coincidencias = []
for fila in tx_h.orderBy("transaccion_id").limit(3).collect():
    en_python = hash_fila_python([fila[c] for c in columnas_hash])
    igual_python = fila["hash_fila"] == en_python
    igual_referencia = fila["hash_fila"] == ESPERADO.get(fila["transaccion_id"])
    coincidencias.append(igual_python and igual_referencia)
    print(fila["transaccion_id"], fila["hash_fila"][:16] + "…",
          "| igual a hash_fila_python:", igual_python, "| igual al valor de referencia:", igual_referencia)
hashes_distintos = tx_h.select("hash_fila").distinct().count()
print("hashes distintos:", hashes_distintos, "de", r_tx["total"])
verificaciones["hash_tres_primeras_ok"] = all(coincidencias) and len(coincidencias) == 3
verificaciones["hashes_distintos"] = hashes_distintos

In [ ]:
escribir_bitacora(spark, f"demo_libreria_{p_variante}", "OK", filas=r_tx["total"],
                  mensaje=json.dumps({"tiempos": tiempos, "verificaciones": verificaciones}))
display(leer_bitacora(spark, ultimas=3))

In [ ]:
resumen = {
    "variante": p_variante,
    "version_funciones": version(),
    "validacion_fechas_ok": r_tx["ok"],
    "filas": r_tx["total"],
    "tiempos": tiempos,
    "verificaciones": verificaciones,
}
print(json.dumps(resumen, indent=2, ensure_ascii=False))